# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. This guide demonstrates loading, exploring, and processing the dataset using its Croissant description.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print the Dataset Title & Description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values to understand data organization. This is key for referencing and extracting data using `mlcroissant`.

In [ ]:
# List available record sets, their @id, and associated fields
print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"- Record set name: {record_set.name}, @id: {record_set.id}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("    Fields:")
        for field in record_set.fields:
            print(f"        - Field: {field.name}, @id: {field.id}, data_type: {field.data_type if hasattr(field, 'data_type') else 'N/A'}")
    if hasattr(record_set, 'columns') and record_set.columns:
        print("    Columns:")
        for col in record_set.columns:
            print(f"        - Column: {col.name}, @id: {col.id}, source: {col.source}")
    print()

### Explore Example Records
Let's print out sample records for a selected record set, referencing it by its `@id`.

In [ ]:
# Choose a record set @id for inspection, typically the main data table. If in doubt, select the first record set.
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
main_record_set_id = record_set_ids[0]

print(f"Sample records from record set @id: {main_record_set_id}")
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    print(json.dumps(rec, indent=2))
    if i >= 2:  # only display a few for inspection
        break

## 3. Data Extraction
Load data from each record set into a DataFrame for programmatic analysis, using only their `@id` values.

In [ ]:
# Extract all record sets into dataframes for further processing
dataframes = {}
for rs in dataset.metadata.record_sets:
    rs_id = rs.id
    print(f"Loading records from record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
        print(f"First records:\n{dataframes[rs_id].head()}\n")
    else:
        print(f"No records found for {rs_id}\n")# Work with the main record set
main_df = dataframes[main_record_set_id]print(f"Fields in main record set DataFrame:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filter, normalize, and group using only field or column `@id`s. Modify the variables below to choose which numeric and group fields to use, referencing their `@id` values printed above.

In [ ]:
# Example: Find a numeric field (column) and a categorical/grouping field
all_fields = list(main_df.columns)
# Choose 'Abundance', 'MW', 'pI', or similar as numeric field (reference its @id from the overview above)
# Choose a categorical field such as 'Sample' or 'Description' or a group field by inspection
# For demonstration, we'll auto-detect numeric columns
numeric_candidates = main_df.select_dtypes(include=['number']).columns.tolist()
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    # If all data is loaded as object, try to convert columns
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col])
            numeric_field_id = col
            break
        except Exception:
            continue

print(f"Using numeric field (by @id): {numeric_field_id}")

# Set an arbitrary threshold for demonstration
threshold = main_df[numeric_field_id].mean() if numeric_field_id in main_df.columns else 10
# Filter records
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a field (choose from non-numeric columns)
from pandas.api.types import is_numeric_dtype
group_candidates = [col for col in main_df.columns if not is_numeric_dtype(main_df[col])]
if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by field @id: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print("Grouped data:")
    print(grouped_df.head())
else:
    print('No categorical field found for grouping.')

## 5. Visualization
Visualize distributions or relationships between fields/columns using their `@id` values. Here, we plot the distribution of the selected numeric field and its grouping.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping field is available, plot mean values
if 'grouped_df' in locals():
    plt.figure(figsize=(10,6))
    sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mass spectrometry dataset with `mlcroissant`. The notebook demonstrated:
- How to reference and extract data by record set and field `@id`.
- How to filter, normalize, and group records referencing relevant field identifiers.
- Methods for visualizing numeric distributions and group effects.

With all dataset elements referenced by their global `@id`, your workflow will remain robust and portable for evolving data schemas.